In [ ]:
# ===============================================================
#   WEEK 8 — GBIKE RENTAL MDP (JACK'S CAR RENTAL)
#   FULL LAB 08 SOLUTION — PROBLEM (2) + PROBLEM (3)
#   Optimized for Google Colab (FAST VALUE ITERATION)
# ===============================================================

import numpy as np
import math
import matplotlib.pyplot as plt
from functools import lru_cache

# ------------------------------------------------------------
#   MDP FORMULATION (As required in lab)
# ------------------------------------------------------------
#   STATE:
#     s = (b1, b2)
#     where b1 = bikes at location 1
#           b2 = bikes at location 2
#     b1, b2 ∈ {0, 1, ..., 20}
#
#   ACTION:
#     a ∈ {-5, ..., +5}
#     a > 0 means move a bikes from loc1 -> loc2
#     a < 0 means move |a| bikes from loc2 -> loc1
#
#   REWARD:
#     R(s, a, s') = 10 × (# rentals fulfilled) - cost(move) - cost(parking)
#
#   TRANSITION:
#     Poisson requests + returns at each location
#     Requests:  ~ Poisson(3), Poisson(4)
#     Returns:   ~ Poisson(3), Poisson(2)
#
#   DISCOUNT:
#     γ = 0.9
#
# ------------------------------------------------------------
# LAB PARAMETERS
# ------------------------------------------------------------
MAX_BIKES   = 20
MAX_MOVE    = 5
RENT_PRICE  = 10
MOVE_COST   = 2
DISCOUNT    = 0.9

# Poisson rates
REQ1, REQ2 = 3, 4
RET1, RET2 = 3, 2

# For speed (truncate Poisson)
POIS_MAX = 6

# Problem (3) modifications
FREE_SHUTTLE    = 1   # 1 free bike from loc1→loc2
PARK_THRESHOLD  = 10
PARK_COST       = 4

# ------------------------------------------------------------
#   FAST POISSON CALC
# ------------------------------------------------------------
@lru_cache(None)
def poisson(n, lam):
    return math.exp(-lam) * (lam**n) / math.factorial(n)

# ------------------------------------------------------------
#  TRANSITION MODEL WITH AND WITHOUT PROBLEM (3) MODIFICATION
# ------------------------------------------------------------
@lru_cache(None)
def transition(state, action, modified=False):
    """
    Return dictionary:
        { next_state : (probability, expected_reward) }
    Transition uses truncated Poisson for speed.
    """
    b1, b2 = state

    # ------------ APPLY OVERNIGHT MOVE ------------
    if action > 0:  # loc1 → loc2
        free = FREE_SHUTTLE if modified else 0
        cost = MOVE_COST * max(0, action - free)
        nb1 = max(0, b1 - action)
        nb2 = min(MAX_BIKES, b2 + action)
    else:           # loc2 → loc1
        cost = MOVE_COST * abs(action)
        nb1 = min(MAX_BIKES, b1 + abs(action))
        nb2 = max(0, b2 - abs(action))

    # ------------ PARKING PENALTIES (Problem 3) ------------
    if modified:
        if nb1 > PARK_THRESHOLD: cost += PARK_COST
        if nb2 > PARK_THRESHOLD: cost += PARK_COST

    transitions = {}
    expected_reward = -cost

    # -------------- POISSON REQUESTS & RETURNS --------------
    for req1 in range(POIS_MAX):
        p_r1 = poisson(req1, REQ1)
        rentals1 = min(req1, nb1)
        rem1 = nb1 - rentals1

        for req2 in range(POIS_MAX):
            p_r2 = poisson(req2, REQ2)
            rentals2 = min(req2, nb2)
            rem2 = nb2 - rentals2
            reward = RENT_PRICE * (rentals1 + rentals2)

            for ret1 in range(POIS_MAX):
                p_ret1 = poisson(ret1, RET1)
                fb1 = min(MAX_BIKES, rem1 + ret1)

                for ret2 in range(POIS_MAX):
                    p_ret2 = poisson(ret2, RET2)
                    fb2 = min(MAX_BIKES, rem2 + ret2)

                    prob = p_r1 * p_r2 * p_ret1 * p_ret2
                    s2 = (fb1, fb2)

                    if s2 not in transitions:
                        transitions[s2] = [0, 0]

                    transitions[s2][0] += prob
                    transitions[s2][1] += prob * reward

    return transitions, expected_reward


# ------------------------------------------------------------
# VALUE ITERATION (FAST & CONVERGENT)
# ------------------------------------------------------------
def value_iteration(modified=False):
    V = np.zeros((MAX_BIKES+1, MAX_BIKES+1))
    policy = np.zeros_like(V, dtype=int)

    while True:
        stable = True
        for b1 in range(MAX_BIKES+1):
            for b2 in range(MAX_BIKES+1):
                best_val = -1e18
                best_act = 0

                for a in range(-MAX_MOVE, MAX_MOVE+1):

                    # check feasibility
                    if not (0 <= b1 - a <= MAX_BIKES): continue
                    if not (0 <= b2 + a <= MAX_BIKES): continue

                    (trans, move_reward) = transition((b1, b2), a, modified)

                    val = move_reward
                    for (n1,n2),(p,rp) in trans.items():
                        val += p*(rp + DISCOUNT*V[n1,n2])

                    if val > best_val:
                        best_val = val
                        best_act = a

                if best_act != policy[b1,b2]:
                    stable = False

                policy[b1,b2] = best_act
                V[b1,b2] = best_val

        if stable:
            break

    return V, policy


# ============================================================
# RUN PROBLEM (2): STANDARD GBIKE
# ============================================================

print("Solving Problem (2): Standard Gbike MDP...")
V2, P2 = value_iteration(modified=False)
print("Done.")

plt.figure(figsize=(8,6))
plt.imshow(P2, origin="lower")
plt.colorbar()
plt.title("Optimal Policy — Problem (2)")
plt.xlabel("Bikes at Location 2")
plt.ylabel("Bikes at Location 1")
plt.show()


# ============================================================
# RUN PROBLEM (3): MODIFIED GBIKE (FREE SHUTTLE + PARK COST)
# ============================================================

print("Solving Problem (3): Modified Gbike MDP...")
V3, P3 = value_iteration(modified=True)
print("Done.")

plt.figure(figsize=(8,6))
plt.imshow(P3, origin="lower")
plt.colorbar()
plt.title("Optimal Policy — Problem (3)")
plt.xlabel("Bikes at Location 2")
plt.ylabel("Bikes at Location 1")
plt.show()

